# Comparing more simulations, boundary conditions

This notebook shows how to compare more TDSE simulations in one notebook and shows the effect of absorbing boundaries.

First, we load the necessary modules and also the TDSE library.

In [ ]:
## python modules used within this notebook
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation
import matplotlib.colors as colors
import os
import sys
import mynumerics as mn
import HHG
from IPython.display import display, Markdown
from IPython.display import HTML
matplotlib.rcParams['animation.embed_limit'] = 200.

from PythonTDSE import *
path_to_DLL = os.path.join(os.environ['TDSE_1D_BUILD'],'libsingleTDSE.so')
DLL = TDSE_DLL(path_to_DLL)

## TDSE with a custom input

Next we define the input field, here we use the same field as in [the introductory tutorial](./analytic_chirped_pulse.ipynb). The field is defined analytically, but samplet to be passed to the solver. We also define the target gas.

In [ ]:
omega0 = mn.ConvertPhoton(1000e-9, 'lambdaSI', 'omegaau')
chirp = 2e-4
E_0 = 0.15  # peak electric field amplitude

T0 = mn.ConvertPhoton(omega0, 'omegaau', 'T0au')
T_max = 3 * T0
N_t = 10000

tgrid = np.linspace(0, T_max, N_t)
E = E_0 * np.sin(np.pi * tgrid / T_max)**2 * np.cos(omega0 * tgrid + chirp * tgrid**2)

gas = 'Ar'

Here we define the parameters for the three simulations, we already provide some comments regarding the results visualised at the end of the notebook. We experiment with no absorber and two absorbers.

### No absorber, various spatial extents
First, we use no absorber, `absorber{'type': 0}`, and various spatial extents given by the number of points: `num_r=6000`, `num_r=1000`, `num_r=250`.

We can observe the differences in spectra and wavefunctions. The spectra are already a good indicator.The first spectrum is a reference, the second spectrum also preserves the main features, but some noise starts to form after 2 a.u. The third spectrum is noisy in the full range. This becomes apparent inspecting the wavefunction. Version 2 reaches the boundary of the simulation box at about 300 a.u., reflects and interferes with the result. For Version 3, the reflections occurs already at 150 a.u. and the whole solution is strongly affected by that afterwards, making the result irrelevant.

### Various absorbers
Second, we use absorbers near the boundaries of the simulation box. There are two absorbers
* *The complex absorber* defined by the options `num_r = 300` or `num_r = 400` and `absorber{'type': 0, 'alpha': 0.001, 'x_cap': 50.}`, it clamps the function by $\exp(-\alpha (|x-x_b|-x_{\mathrm{CAP}})^2)$, where $x_b=x_{\mathrm{min}/\mathrm{max}}$ is the boundary of the simulation box$^\dagger$.
*  The smoothestep function, `num_r = 750` and `absorber{'type': 0, 'x_cap': 140.}`, defined by by $3x^2 - 2x^3$ rescaled to the intervals $[x_b, x_{\mathrm{CAP}}]$ and $[ x_{\mathrm{CAP}},x_b]$.

Comment and uncomment the respective parts of the input to run TDSE's again with the absorbers. All the results are then free of strong noise and wavefunction is efficiently damped (investigate this figure in more detail) near the boundaries. Let us focus on the spectra. The results for both absorbers are similar to each other (using `num_r = 300` for `version 2`) and differ from the reference result, the reason is that the long trajectories are filtered out, [see this notebook for further discussion](./field_from_CUPRAD.ipynb). Note that the extent of the smoothstep absorber is much larger due to the reason that it goes to 0 in the clamped region, while the complex absorber reaches only about 73 % absorption at the hard boundary (for the template parameters). It shows that it is sufficient to clamp the wavefunction gently as the absorbing effect accumulates with consecutive time steps. With the second option, `num_r = 400`, the spectrum is close to the reference spectrum in the plateau region, but the base level of noise (linked with the precision) is higher deeper in the cutoff and beyond.

These experiments intorduces basic features of the absorbers and allows further experiments with the code.

$^\dagger$ It is derived from the additional complex part of the potential in the Hamiltonian $H_{\mathrm{CAP}}(t) = H(t) + V_{\mathrm{CAP}}$, where $V_{\mathrm{CAP}}=-{\bf{i}}\alpha (|x-x_b|-x_{\mathrm{CAP}})^2)$ with $x_b=x_{\mathrm{min}/\mathrm{max}}$ being the boundary of the simulation box and the potential is applied only in the region near the boundary.


In [ ]:
inputs1 = inputs_def()
inputs1.init_default_inputs(
    Eguess=-HHG.Ip_list[gas],
    trg_a = HHG.soft_Coulomb_a[gas],
    dt=0.125,
    dx=0.4,
    num_r=6000,
    writewft=1,
    tprint=1.,
    x_int=2.,
    absorber={'type': 0}
)
inputs2 = inputs_def()
inputs2.init_default_inputs(
    Eguess=-HHG.Ip_list[gas],
    trg_a = HHG.soft_Coulomb_a[gas],
    dt=0.125,
    dx=0.4,
    num_r=1000,
    # num_r=300,
    # num_r=400,
    writewft=1,
    tprint=1.,
    x_int=2.,
    absorber={'type': 0},
    # absorber={
    #     'type': 1,
    #     'x_cap': 50.,
    #     'alpha': 0.001
    # },
)
inputs3 = inputs_def()
inputs3.init_default_inputs(
    Eguess=-HHG.Ip_list[gas],
    trg_a = HHG.soft_Coulomb_a[gas],
    dt=0.125,
    dx=0.4,
    num_r=250,
    # num_r=750,
    writewft=1,
    tprint=1.,
    x_int=2.,
    absorber={'type': 0},
    # absorber={
    #     'type': 2,
    #     'x_cap': 140.
    # },
)

print('Version 1: xmax =', 0.5 * inputs1.num_r * inputs1.dx)
print('Version 2: xmax =', 0.5 * inputs2.num_r * inputs2.dx)
print('Version 3: xmax =', 0.5 * inputs3.num_r * inputs3.dx)

# print('damping factor at the boundary:', np.exp(-0.001*50.**2 * 0.125))

Here we initialise the inputs, run the simulations, and store the outputs into a list for further processing.

(Note that we evaluate TDSE's directl before constructing the list. The reason is that the interface intrisically uses `c_types` module, which allocates memory by low-level routines, and we encoutered some unexpected behavour if the calls were applied on already nested structures.)

In [ ]:
inputs1.init_time_and_field(DLL, E=E, t=tgrid)
DLL.init_GS(inputs1)

outputs1 = outputs_def()
DLL.call1DTDSE(inputs1, outputs1)

inputs2.init_time_and_field(DLL, E=E, t=tgrid)
DLL.init_GS(inputs2)

outputs2 = outputs_def()
DLL.call1DTDSE(inputs2, outputs2)

inputs3.init_time_and_field(DLL, E=E, t=tgrid)
DLL.init_GS(inputs3)

outputs3 = outputs_def()
DLL.call1DTDSE(inputs3, outputs3)


TDSEs = [
    {
        'inputs': inputs1,
        'outputs': outputs1,
    },
    {
        'inputs': inputs2,
        'outputs': outputs2,
    },
    {
        'inputs': inputs3,
        'outputs': outputs3,
    },
]

Finally, we plot the results with the parameters defined below.

In [ ]:
omega_max_plot = 5.5 # [a.u.]

# Optional spatial filtering
filter_xgrid = False
x_max_plot = 250.1  # [a.u.]

# Optional filtering of small wavefunction values
filter_wavefunction_min = False
wavefunction_min = 1e-8
wavefunction_max = 0.5


# -------------------------------------------------------------------------
# 1. Electric field
# -------------------------------------------------------------------------

fig1, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(
    TDSEs[0]['outputs'].get_tgrid(),
    TDSEs[0]['outputs'].get_Efield(),
    label='Electric field'
)

ax1.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax1.set_ylabel(r'$\mathcal{E}~[\mathrm{a.u.}]$')
ax1.legend()

fig1.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 2. Harmonic spectra
# -------------------------------------------------------------------------

fig2, ax2 = plt.subplots(figsize=(8, 5))
photon_energy_ranges = []

for k, tdse in enumerate(TDSEs):
    ogrid = tdse['outputs'].get_omegagrid()
    ko_max = mn.FindInterval(ogrid, omega_max_plot)

    photon_energy = ogrid[:ko_max]
    spectrum = np.abs(tdse['outputs'].get_Fsourceterm())[:ko_max]

    ax2.semilogy(photon_energy, spectrum, label=f'Version {k + 1}')
    photon_energy_ranges.append((photon_energy.min(), photon_energy.max()))

ax2.set_xlim(
    min(limits[0] for limits in photon_energy_ranges),
    max(limits[1] for limits in photon_energy_ranges)
)

ax2.set_xlabel(r'$\omega~[\mathrm{a.u.}]$')
ax2.set_ylabel(
    r'$|(\partial \hat{\jmath}/\partial t)(\omega)|'
    r'~[\mathrm{arb.~u.}]$'
)
ax2.legend()

fig2.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 3. Wavefunctions
# -------------------------------------------------------------------------

wavefunction_data = []

for tdse in TDSEs[:3]:
    t_psi, x_grid, wavefunction = tdse['outputs'].get_wavefunction(
        tdse['inputs'],
        grids=True
    )

    x_range = np.abs(x_grid) < x_max_plot if filter_xgrid else slice(None)
    psi_plot = np.abs(wavefunction).T[x_range]

    if filter_wavefunction_min:
        psi_plot = np.maximum(psi_plot, wavefunction_min)

    wavefunction_data.append((t_psi, x_grid[x_range], psi_plot))


# Common limits covering all three plots
t_min = min(t_psi.min() for t_psi, x_grid, psi_plot in wavefunction_data)
t_max = max(t_psi.max() for t_psi, x_grid, psi_plot in wavefunction_data)
x_min = min(x_grid.min() for t_psi, x_grid, psi_plot in wavefunction_data)
x_max = max(x_grid.max() for t_psi, x_grid, psi_plot in wavefunction_data)

wavefunction_norm = colors.LogNorm(
    vmin=wavefunction_min,
    vmax=wavefunction_max
)

fig3, axes = plt.subplots(
    1,
    3,
    figsize=(10, 5.5),
    sharex=True,
    sharey=True,
    layout='constrained'
)

for k, (ax, (t_psi, x_grid, psi_plot)) in enumerate(
    zip(axes, wavefunction_data)
):
    pc = ax.pcolormesh(
        t_psi,
        x_grid,
        psi_plot,
        cmap='jet',
        norm=wavefunction_norm,
        shading='auto'
    )

    ax.set_title(f'Version {k + 1}')
    ax.set_xlabel(r'$t~[\mathrm{a.u.}]$')

axes[0].set_ylabel(r'$x~[\mathrm{a.u.}]$')
axes[0].set_xlim(t_min, t_max)
axes[0].set_ylim(x_min, x_max)

cbar = fig3.colorbar(
    pc,
    ax=axes,
    orientation='horizontal',
    location='bottom',
    pad=0.08,
    shrink=0.8,
    aspect=45
)

cbar.set_label(r'$|\psi|~[\mathrm{a.u.}]$')

plt.show()